In [1]:
import sys
sys.path.append("src")

import numpy as np
from pipeline import run_pulse_sort

npx_path = "data/raw/c37_npx_raw.bin"
npx_recording = np.memmap(npx_path, mode='r', dtype=np.int16, order='C')
npx_recording = npx_recording.reshape((384, 4050171), order='F')

channels_subset = list(range(180, 220))
subset_data = npx_recording[channels_subset, :].T.astype(np.float32)
fs = 30000

result = run_pulse_sort(subset_data, fs)
print(f"Spike times: {result.spike_times.shape}")
print(f"Cluster labels: {np.unique(result.cluster_labels)}")
print(f"Number of templates: {len(result.templates)}")

Spike times: (2685,)
Cluster labels: [0 1 2 3 4 5 6 7 8 9]
Number of templates: 10


In [2]:
import sys
sys.path.append("src")
sys.path.append("src/spikeinterface_pulse_sort")

import os
import numpy as np
from spikeinterface.core import NumpyRecording
from pulsesortsorter import PulseSortSorter

npx_path = "data/raw/c37_npx_raw.bin"
npx_recording_raw = np.memmap(npx_path, mode='r', dtype=np.int16, order='C')
npx_recording_raw = npx_recording_raw.reshape((384, 4050171), order='F')

channels_subset = list(range(180, 220))
subset_data = npx_recording_raw[channels_subset, :].T.astype(np.float32)
fs = 30000

recording = NumpyRecording(traces_list=[subset_data], sampling_frequency=fs)

output_folder = "pulse_sort_output"
os.makedirs(output_folder, exist_ok=True)

params = PulseSortSorter._default_params.copy()

PulseSortSorter._setup_recording(recording, output_folder, params, verbose=True)
PulseSortSorter._run_from_folder(output_folder, params, verbose=True)
sorting = PulseSortSorter._get_result_from_folder(output_folder)

print(sorting)
print(f"Number of units: {len(sorting.get_unit_ids())}")

NumpySorting: 10 units - 1 segments - 30.0kHz
Number of units: 10
